In [12]:
from tqdm import tqdm
import torch.nn as nn
import torch
from safetensors.torch import load_file  # comes with HF if safetensors installed
import time
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
import numpy as np, time
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from scipy.special import softmax
import os
import glob
from typing import List
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")

from NACE_helper import NACE_code_structure, NACE_helper
import pandas as pd

np.random.seed(7)

In [2]:
sections_full = NACE_code_structure.level_1 
divisions_full = NACE_code_structure.level_2
classes_full = NACE_code_structure.level_3

In [4]:
# ------------------------------
# 1) Hierarchy
# ------------------------------
sections = {"A": "Agriculture", "B": "Mining"}# "C": "Manufacturing", "K": "Financials"
divisions = {
    "A": ["1", "2", "3"],  
    "B": ['5', '6', '7'],
    "C": ['20', '21'], 
    "F": ["41", "42", "43"], 
    "J": ["58", "63"],
    }
classes = {
    "1": ["01.1","01.2","01.3","01.4"],
    "2": ['02.1', '02.2', '02.3'],
    "3": ['03.1', '03.2'],
    "5": ['05.1', '05.2'],
    "6": ['06.1', '06.2'],
    "7": ['07.1', '07.2'],
    "20": ['20.1', '20.2', '20.3', '20.4', '20.5', '20.6'],
    "21": ['21.1', '21.2'],
    "41": ["41.1", "41.2"], 
    "42": ["42.1", "42.2","42.9"], 
    "43": ["43.1", "43.2", "43.3","43.9"], 
    "58": ['58.1', '58.2'],
    "63": ['63.1', '63.9'],
    }

division_of_class = {}
section_of_division = {}
all_divisions = []
for sec, divs in divisions.items():
    all_divisions.extend(divs)
    for d in divs:
        section_of_division[d] = sec
        for c in classes[d]:
            division_of_class[c] = d
all_classes = list(division_of_class.keys())

In [5]:
nace_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")

get_description = lambda code: nace_descriptions[nace_descriptions["CODE"] == code]["NAME"].iloc[0]

#### **Load real data**

In [ ]:


df_overview_2_with_description = pd.read_csv("data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview_with_description.csv", sep=";")
df_overview_2_with_description = df_overview_2_with_description[df_overview_2_with_description["Description_clean"].notna()]
#df_overview_2_with_description = df_overview_2_with_description[df_overview_2_with_description["NACE_letter"].isin(sections.keys())]
#df_overview_2_with_description = df_overview_2_with_description[df_overview_2_with_description["NACE_lvl_3"].isin(list(map(float, all_classes)))]
df_overview_2_with_description

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Symbol,Description_page,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,...,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2,Description,Description_clean,Tested_Class
2,2,2,2,ID1000167901,NaN,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,...,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,3.1,3,## Riwayat Singkat Perseroan\n\nThe Company at...,PT Dharma Samudera Fishing Industries Tbk or '...,True
11,11,11,11,LU0611262873,NaN,KSG Agro SA,1,2010.0,POL,L5903L107,...,1,SHARE,B42XFB0,A,KSG Agro SA1.pdf,1.5,1,"## PRINCIPAL ACTIVITIES\n\nKSG Agro S.A., sepa...","KSG Agro S.A., together with its subsidiaries,...",False
21,21,21,21,AU000000CSS3,NaN,Clean Seas Seafood Limited,1,2000.0,AUS,Q2508T119,...,1,SHARE,B0PFW92,A,Clean Seas Seafood Limited2.pdf,3.2,3,For personal use only\n\n<!-- image -->\n\n## ...,Clean Seas is the global leader in the full cy...,True
85,85,85,85,KR7086060001,5.0,"Gene Bio Tech Co., Ltd.",1,2000.0,KOR,Y2684U105,...,1,SHARE,B11R103,A,"Gene Bio Tech Co., Ltd.1.pdf",1.6,1,## WHO WE ARE\n\nBio-Gene is an Australian ...,Bio-Gene is an Australian agtech development c...,False
95,95,95,95,CA89154B1022,NaN,Total Energy Services Inc.,1,1996.0,CAN,89154B102,...,0,SHARE,BF2HFP3,B,Total Energy Services Inc.3.pdf,9.1,9,NaN,1.1 TotalEnergies at a glance\n1.1.1 A multi-e...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,1202,1202,1202,BMG8766E1093,NaN,Textainer Group Holdings Limited,1,1979.0,USA,G8766E109,...,0,SHARE,BKDZ8P2,N,Textainer Group Holdings Limited1.pdf,77.3,77,<!-- image -->\n\n## About Us\n\nTextainer Gro...,Textainer Group Holdings Limited (NYSE: TGH) i...,False
1223,1223,1223,1223,AU000000MMS5,NaN,Mcmillan Shakespeare Limited,1,1988.0,AUS,Q58998107,...,1,SHARE,B00G1Q0,N,Mcmillan Shakespeare Limited1.pdf,77.1,77,MMS ANNUAL REPORT 2022\n\nB\n\n## Annual Gene...,The McMillan Shakespeare Group is a provider o...,False
1238,1238,1238,1238,US10948C1071,NaN,"BrightView Holdings, Inc.",1,2013.0,USA,10948C107,...,1,SHARE,BG0ZML1,N,"BrightView Holdings, Inc.1.pdf",81.3,81,## General Risk Factors\n\n- Natural disasters...,"BrightView Holdings, Inc. is a holding company...",False
1241,1241,1241,1241,US23204X1037,NaN,Custom Truck One Source Inc,1,1996.0,USA,23204X103,...,1,SHARE,BL66YS4,N,Custom Truck One Source Inc1.pdf,77.3,77,## Item 1. Business\n\n## Company Overview...,"Custom Truck One Source, Inc. is engaged in th...",False


In [7]:
# Data 1

# df_val_3 = []

# for path in [
#     "data/synthetic_data/data_20251221__level_3__subclasses_1",
#     "data/synthetic_data/data_20251221__level_3__subclasses_2",
#     "data/synthetic_data/data_20251221__level_3__subclasses_3",
#     "data/synthetic_data/data_20251221__level_3__subclasses_5",
#     "data/synthetic_data/data_20251221__level_3__subclasses_6",
#     "data/synthetic_data/data_20251221__level_3__subclasses_7",
# ]: 

#     df_val_3.append(pd.read_csv(path+"/val_data.csv"))

# df_val_3 = pd.concat(df_val_3, ignore_index=True)
# #df_val_3["label"] = df_val_3["label"].astype(str).apply(lambda x: "0" + x if len(x) == 3 else x)
# df_val_3["label"] = df_val_3["label"].astype(str)

# df_val_2 = []

# for path in [
#     "data/synthetic_data/data_20251222__level_2__subclasses_A__level_descriptions_3",
#     "data/synthetic_data/data_20251222__level_2__subclasses_B__level_descriptions_3"
# ]: 

#     df_val_2.append(pd.read_csv(path+"/val_data.csv"))

# df_val_2 = pd.concat(df_val_2, ignore_index=True)
# df_val_2["label"] = df_val_2["label"].astype(str)
# df_val_2 = df_val_2[df_val_2["label"].apply(lambda x: x in list(all_divisions))]

# dataset_path_1 = "data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3/val_data.csv"
# df_val_1 = pd.read_csv(dataset_path_1)

#### Load synthetic data

In [8]:
# Data 2

df_val_3 = []

for path in [
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_1__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_2__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_3__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_5__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_6__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_7__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_20__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_21__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_41__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_42__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_43__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_58__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260120__level_3__subclasses_63__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
]: 

    df_val_3.append(pd.read_csv(path+"/val_data.csv"))

df_val_3 = pd.concat(df_val_3, ignore_index=True)
#df_val_3["label"] = df_val_3["label"].astype(str).apply(lambda x: "0" + x if len(x) == 3 else x)
df_val_3["label"] = df_val_3["label"].astype(str)

df_val_2 = []

for path in [
    "data/synthetic_data/two_step/data_20260119__level_2__subclasses_A__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260119__level_2__subclasses_B__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260119__level_2__subclasses_C__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260119__level_2__subclasses_F__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
    "data/synthetic_data/two_step/data_20260119__level_2__subclasses_J__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini",
]: 

    df_val_2.append(pd.read_csv(path+"/val_data.csv"))

df_val_2 = pd.concat(df_val_2, ignore_index=True)
df_val_2["label"] = df_val_2["label"].astype(str)
df_val_2 = df_val_2[df_val_2["label"].apply(lambda x: x in list(all_divisions))]

dataset_path_1 = "data/synthetic_data/two_step/data_20260119__level_1__subclasses_None__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-minifull/val_data.csv"
df_val_1 = pd.read_csv(dataset_path_1)

df_val_1 = df_val_1[df_val_1["label"].apply(lambda x: x in list(sections.keys()))]

X_lvl_1_cal = df_val_1["text"].to_numpy()
y_lvl_1_cal = df_val_1["label"].to_numpy()

X_lvl_2_cal = df_val_2["text"].to_numpy()
y_lvl_2_cal = df_val_2["label"].to_numpy()

X_lvl_3_cal = df_val_3["text"].to_numpy()
y_lvl_3_cal = df_val_3["label"].to_numpy()

# X_lvl_3_test and y_lvl_3_test is the data to test
X_lvl_3_cal, X_lvl_3_test, y_lvl_3_cal, y_lvl_3_test = train_test_split(X_lvl_3_cal, y_lvl_3_cal, test_size=0.3, random_state=0, stratify=y_lvl_3_cal)

len(X_lvl_1_cal), len(X_lvl_2_cal), len(X_lvl_3_cal)

(603, 3193, 5961)

In [ ]:
# def build_tree_str(sections, divisions, classes, indent="  "):
#     lines = []
#     for sec_code, sec_name in sorted(sections.items()):
#         try: 
#             num_sentences = df_statistics_1[df_statistics_1["NACE_Code"] == sec_code]["Sentences"].iloc[0]
#         except IndexError:
#             num_sentences = 0

#         lines.append(f"{sec_code} — {sec_name}: {num_sentences} training samples")

#         for div_code in sorted(divisions.get(sec_code, []), key=lambda x: (len(x), x)):
#             try: 
#                 num_sentences = df_statistics_2[df_statistics_2['NACE_Code'] == div_code]['Sentences'].iloc[0]
#             except IndexError:
#                 num_sentences = 0
#             lines.append(f"{indent}{div_code}: {get_description(div_code)} {num_sentences}")

#             for cls_code in sorted(classes.get(div_code, []), key=lambda x: tuple(map(int, x.split(".")))):
#                 try: 
#                     num_sentences = df_statistics_3[df_statistics_3['NACE_Code'] == cls_code]['Sentences'].iloc[0]
#                 except IndexError:
#                     num_sentences = 0
#                 lines.append(f"{indent}{indent}{cls_code}: {get_description(cls_code)} {num_sentences}")
#     return "\n".join(lines)


# print(build_tree_str(sections, divisions, classes))

# print(build_tree_str(sections_full, divisions_full, classes_full))

In [9]:
f = lambda x : "0"+x
y_lvl_3_cal_lvl_2 = np.vectorize(lambda x: NACE_helper.get_all_level(f(x))[2])(y_lvl_3_cal)
y_lvl_3_cal_lvl_2

array(['1', '2', '2', ..., '5', '5', '2'], dtype='<U1')

In [10]:
y_lvl_2_cal_lvl_1 = np.vectorize(NACE_helper.get_level_1_nace)(y_lvl_2_cal)
y_lvl_2_cal_lvl_1

array(['A', 'A', 'A', ..., 'J', 'J', 'J'], dtype='<U1')

In [21]:
# ------------------------------
# 2) Import BERT Models
# ------------------------------

def load_custom_bert_from_checkpoint(ckpt_path: str, num_layers_base: int = 1, num_labels_base: int = None):
    # 1. Load the saved config (includes custom_hidden, custom_num_layers)
    config = AutoConfig.from_pretrained(ckpt_path)

    full_config = AutoConfig.from_pretrained(ckpt_path.replace(os.path.basename(ckpt_path), "training_config.json"))

    # 2. Build a model from config (bare BertForSequenceClassification)
    model = AutoModelForSequenceClassification.from_config(config)

    # 3. Rebuild the SAME classifier architecture as in training
    hidden = getattr(config, "custom_hidden", 512)  # fallback if not in config
    num_layers = getattr(full_config, "num_layers", num_layers_base)
    num_labels = getattr(config, "num_labels", num_labels_base)

    layers = []
    for i in range(num_layers):
        in_dim = config.hidden_size if i == 0 else hidden
        layers.append(nn.Linear(in_dim, hidden))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(0.2))

    layers.append(nn.Linear(hidden, num_labels))
    model.classifier = nn.Sequential(*layers)

    # 4. Load weights from model.safetensors
    state_dict = load_file(os.path.join(ckpt_path, "model.safetensors"))
    model.load_state_dict(state_dict, strict=True)  # will fail loudly if mismatch

    # 5. Inference mode
    model.eval()
    return model


ckpt_path_sec = "results/BERT_models/NACE_synthetic_data/006_results__synthetic_data_1__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-380"
ckpt_path_sec = "results/BERT_models/NACE_synthetic_data/106_results__level_1__subclasses_None__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-minifull__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-684"

sec_clf = load_custom_bert_from_checkpoint(ckpt_path_sec)

# ckpt_path_div = {
#     #"A":"results/BERT_models/NACE_synthetic_data/062_results__level_2__subclasses_A__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-102", 
#     "A":"results/BERT_models/NACE_synthetic_data/067_results__level_2__subclasses_A____num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-169",
#     "B": "results/BERT_models/NACE_synthetic_data/068_results__level_2__subclasses_B____num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-169",
#     # This one is for K -> [64,65,66]
#     #"K": "results/BERT_models/NACE_classification/NACE_level_2/NACE_class_K/001_results__data_approach_2__num_layers_2__cos_thres_0.4bert-base-uncased__train_full_model__some_labels_no_G__only_labels/checkpoint-634",
#     # This one is for K -> [64,66]
#     #"K": "results/BERT_models/NACE_classification/NACE_level_2/NACE_class_K/002_results__data_approach_2__num_layers_2__cos_thres_0.35bert-base-uncased__train_full_model__some_labels_no_G__only_labels/checkpoint-69",
#     # "F": "results/BERT_models/NACE_classification/NACE_level_2/NACE_class_F/001_results__data_approach_2__num_layers_2__cos_thres_0.35bert-base-uncased__train_full_model__some_labels_no_G__only_labels/checkpoint-260",
#     # "J": "results/BERT_models/NACE_classification/NACE_level_2/NACE_class_J/001_results__data_approach_2__num_layers_2__cos_thres_0.35bert-base-uncased__train_full_model__some_labels_no_G__only_labels/checkpoint-48",
#     # "N": "results/BERT_models/NACE_classification/NACE_level_2/NACE_class_N/001_results__data_approach_2__num_layers_2__cos_thres_0.35bert-base-uncased__train_full_model__some_labels_no_G__only_labels/checkpoint-300", 
# } 
ckpt_path_div = {
    "A": 'results/BERT_models/NACE_synthetic_data/107_results__level_2__subclasses_A__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-226',
    "F": 'results/BERT_models/NACE_synthetic_data/108_results__level_2__subclasses_F__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-904',
    "B": 'results/BERT_models/NACE_synthetic_data/109_results__level_2__subclasses_B__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-442',
    "C": 'results/BERT_models/NACE_synthetic_data/110_results__level_2__subclasses_C__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-390',
    "J": 'results/BERT_models/NACE_synthetic_data/111_results__level_2__subclasses_J__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-300',
} 

div_clf_dict = {k: load_custom_bert_from_checkpoint(v) for k,v in ckpt_path_div.items()}
ckpt_path_cls = {
    "1":'results/BERT_models/NACE_synthetic_data/112_results__level_3__subclasses_1__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-972',
    "2":'results/BERT_models/NACE_synthetic_data/113_results__level_3__subclasses_2__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-119',
    "3":'results/BERT_models/NACE_synthetic_data/114_results__level_3__subclasses_3__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-164',
    "5":'results/BERT_models/NACE_synthetic_data/115_results__level_3__subclasses_5__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-148',
    "6":'results/BERT_models/NACE_synthetic_data/116_results__level_3__subclasses_6__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-149',
    "7":'results/BERT_models/NACE_synthetic_data/117_results__level_3__subclasses_7__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-294',
    "20":'results/BERT_models/NACE_synthetic_data/118_results__level_3__subclasses_20__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-1368',
    "21":'results/BERT_models/NACE_synthetic_data/119_results__level_3__subclasses_21__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-225',
    "41":'results/BERT_models/NACE_synthetic_data/120_results__level_3__subclasses_41__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-150',
    "42":'results/BERT_models/NACE_synthetic_data/121_results__level_3__subclasses_42__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-113',
    "43":'results/BERT_models/NACE_synthetic_data/122_results__level_3__subclasses_43__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-750',
    "58":'results/BERT_models/NACE_synthetic_data/123_results__level_3__subclasses_58__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-154',
    "63":'results/BERT_models/NACE_synthetic_data/124_results__level_3__subclasses_63__prompts_5_list__few_shot__from_lvl_4_topics__gpt-4o-mini__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-228',
} 
# ckpt_path_cls = {
#     "1": "results/BERT_models/NACE_synthetic_data/072_results__level_3__subclasses_1__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-270",
#     "2": "results/BERT_models/NACE_synthetic_data/073_results__level_3__subclasses_2__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-306",
#     "3": "results/BERT_models/NACE_synthetic_data/074_results__level_3__subclasses_3__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-476",
#     "5": "results/BERT_models/NACE_synthetic_data/075_results__level_3__subclasses_5__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-136",
#     "6": "results/BERT_models/NACE_synthetic_data/076_results__level_3__subclasses_6__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-68",
#     "7": "results/BERT_models/NACE_synthetic_data/077_results__level_3__subclasses_7__num_layers_2__model_namebert-base-uncased__train_full_model/checkpoint-68", 
# } 
cls_clf_dict = {k: load_custom_bert_from_checkpoint(v) for k,v in ckpt_path_cls.items()}

# tokenizer is used for all models
tokenizer = AutoTokenizer.from_pretrained(ckpt_path_sec) 

idx_sec = sec_clf.config.label2id
idx_div_dict = {k: v.config.label2id for k, v in div_clf_dict.items()}
idx_cls_dict = {k: v.config.label2id for k, v in cls_clf_dict.items()}

In [22]:
# get all checkpoints, to filter out the right BERT models
all_checkpoints = glob.glob("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/results/BERT_models/NACE_synthetic_data/*/checkpoint*")
list(filter(lambda x: int(x.split("NACE_synthetic_data")[1][1:4]) > 105, all_checkpoints))
1

1

In [23]:
import numpy as np

def softmax(x, temperature=1.0):
    """
    Softmax with temperature.

    Args:
        x (array-like): input logits
        temperature (float): temperature parameter (T > 0)

    Returns:
        np.ndarray: probability distribution
    """
    x = np.asarray(x, dtype=np.float64)

    # scale by temperature
    x = x / temperature

    # numerical stability
    x = x - np.max(x)

    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

In [24]:
TEMPERATURE = 1.0
def BERT_classification_chunk(chunks: List[str], model, tokenizer=tokenizer, temperature=TEMPERATURE, waitbar=False) -> List:

    probas = []

    if waitbar:
        chunks = tqdm(chunks)

    for chunk in chunks: 
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.squeeze(0)

        probas.append(softmax(logits, temperature=temperature))
    return np.array(probas)

In [ ]:
# import matplotlib.pyplot as plt

# for i in [0.1, 0.5, 1.0, 2.0, 5.0]:
#     probas = BERT_classification_chunk(X_lvl_1_cal[:2], model=sec_clf, temperature=i)
#     probas
#     plt.figure()
#     plt.title(f"Temperature: {i}")
#     plt.bar(range(len(probas[0])), probas[0])


In [25]:
# ------------------------------
# 5) RAPS + reject calibration
# ------------------------------
K_FREE = 1
LAMBDA = 0.05

def renorm_rows(v):
    s = v.sum(axis=1, keepdims=True)
    return np.divide(v, s, out=np.full_like(v, 1.0 / v.shape[1]), where=(s > 0))

def raps_score_rowwise_argsort(P, true_idx, k_free=K_FREE, lam=LAMBDA):
    order = np.argsort(-P, axis=1)
    P_sorted = np.take_along_axis(P, order, axis=1)
    true_mask = (order == true_idx[:, None])
    rank_pos = true_mask.argmax(axis=1)   # exact argsort rank, 0-based
    r = rank_pos + 1
    cumsum = np.cumsum(P_sorted, axis=1)
    cum = cumsum[np.arange(P.shape[0]), rank_pos]
    penalty = lam * np.maximum(0, r - k_free)
    return cum + penalty

def reject_score(P_sec):
    return 1.0 - P_sec.max(axis=1)

def qthr(a, eps):
    return float(np.quantile(a, 1.0-eps, method="higher"))

# Predict probabilities on calibration in batch
t0 = time.time()
print(f"Predict {len(X_lvl_1_cal)} Sections.")
P_sec_cal = BERT_classification_chunk(X_lvl_1_cal, sec_clf, waitbar=True)

Predict 603 Sections.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 603/603 [00:17<00:00, 35.23it/s]


In [26]:
print(f"Predict {len(X_lvl_2_cal)} Divisions.")
P_div_cal_dict = {k: BERT_classification_chunk(X_lvl_2_cal, v, waitbar=True) for k, v in div_clf_dict.items()}

Predict 3193 Divisions.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3193/3193 [01:32<00:00, 34.40it/s]


In [ ]:
print(f"Predict {len(X_lvl_3_cal)} Classes.")
P_cls_cal_dict = {k: BERT_classification_chunk(X_lvl_3_cal, v, waitbar=True) for k, v in cls_clf_dict.items()}
t_prob = time.time() - t0

Predict 5961 Classes.


  1%|██▏                                                                                                                                                                            | 76/5961 [00:02<02:54, 33.76it/s]

In [ ]:
# --- thresholds ---
EPS_SEC = EPS_DIV = EPS_CLS = 0.1
EPS_REJECT = 0.15
MAX_LEAF_SIZE = 3

true_sec_idx = np.array([idx_sec[s] for s in y_lvl_1_cal])
raps_sec_scores = raps_score_rowwise_argsort(P_sec_cal, true_sec_idx)
tau_sec = qthr(raps_sec_scores, EPS_SEC)

In [ ]:
# Division threshold: Mondrian-by-parent (true section)
tau_div_dict = {}
for div, _ in div_clf_dict.items():

    #print(f"Get tau for classifier {div}: {idx_div_dict[div].keys()}")

    # Only take correctly chosen sections (lvl 1)
    mask = (y_lvl_2_cal_lvl_1 == div)
    div_scores = np.empty(len(X_lvl_2_cal[mask]))
    if not np.any(mask):
        continue
    
    child = list(idx_div_dict[div].keys())
    child_idx = np.array([idx_div_dict[div][d] for d in child], dtype=int)

    # Probs of the highest scoring class
    P_child = renorm_rows(P_div_cal_dict[div][mask][:, child_idx])
    
    # Right division
    true_div_local = np.array([child.index(d) for d in y_lvl_2_cal[mask]], dtype=int)
    
    # S scores
    div_scores = raps_score_rowwise_argsort(P_child, true_div_local)
    
    # get tau 
    tau_div_temp = qthr(div_scores, EPS_DIV)
    tau_div_dict[div] = tau_div_temp

tau_div_dict

In [ ]:
# Class threshold: Mondrian-by-parent (true division)
tau_cls_dict = {}
for cls_, _ in cls_clf_dict.items(): 
    
    mask = (y_lvl_3_cal_lvl_2 == cls_)
    cls_scores = np.empty(len(X_lvl_3_cal[mask]))
    if not np.any(mask):
        continue
    
    child = list(idx_cls_dict[cls_].keys())
    child_idx = np.array([idx_cls_dict[cls_][c] for c in child], dtype=int)
    
    P_child = renorm_rows(P_cls_cal_dict[cls_][mask][:, child_idx])
    true_cls_local = np.array([child.index(c) for c in y_lvl_3_cal[mask]], dtype=int)

    cls_scores = raps_score_rowwise_argsort(P_child, true_cls_local)
    tau_cls_temp = qthr(cls_scores, EPS_CLS)
    tau_cls_dict[cls_] = tau_cls_temp

tau_cls_dict

In [ ]:
EPS_REJECT=0.1

In [ ]:
# Reject threshold calibrated on in-dist calibration points only
rej_scores_in = reject_score(P_sec_cal)
tau_rej = float(np.quantile(rej_scores_in, 1.0 - EPS_REJECT, method="higher"))
tau_rej

In [ ]:
print("Tau for section:", tau_sec)
print("Tau for each division: ", tau_div_dict)
print("Tau for each class: ", tau_cls_dict)

In [ ]:
# # ------------------------------
# # 6) EXACT set-based metrics (vectorized)
# # ------------------------------
# t0 = time.time()

# rej_cal = (reject_score(P_sec_cal) > tau_rej)
# kept = ~rej_cal

# # Section membership + size
# sec_in_set = (raps_score_rowwise_argsort(P_sec_cal, true_sec_idx) <= tau_sec)
# sec_membership = np.column_stack([
#     (raps_score_rowwise_argsort(P_sec_cal, np.full(len(X_cal), j, dtype=int)) <= tau_sec)
#     for j in range(P_sec_cal.shape[1])
# ])
# sec_set_size = sec_membership.sum(axis=1)

# # Division membership + size (Mondrian true parent)
# div_in_set_mondrian = np.empty(len(X_cal), dtype=bool)
# div_set_size_mondrian = np.empty(len(X_cal), dtype=int)
# for s in sections.keys():
#     mask = (y_sec_cal == s)
#     if not np.any(mask):
#         continue
#     child = divisions[s]
#     child_idx = np.array([idx_div_dict[d] for d in child], dtype=int)
#     P_child = renorm_rows(P_div_cal[mask][:, child_idx])
#     true_div_local = np.array([child.index(d) for d in y_div_cal[mask]], dtype=int)

#     div_in_set_mondrian[mask] = (raps_score_rowwise_argsort(P_child, true_div_local) <= tau_div)
#     memb = np.column_stack([
#         (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_div)
#         for j in range(P_child.shape[1])
#     ])
#     div_set_size_mondrian[mask] = memb.sum(axis=1)

# # Class membership + size (Mondrian true parent)
# cls_in_set_mondrian = np.empty(len(X_cal), dtype=bool)
# cls_set_size_mondrian = np.empty(len(X_cal), dtype=int)
# for d, child in classes.items():
#     mask = (y_div_cal == d)
#     if not np.any(mask):
#         continue
#     child_idx = np.array([idx_cls_dict[c] for c in child], dtype=int)
#     P_child = renorm_rows(P_cls_cal[mask][:, child_idx])
#     true_cls_local = np.array([child.index(c) for c in y_cls_cal[mask]], dtype=int)

#     cls_in_set_mondrian[mask] = (raps_score_rowwise_argsort(P_child, true_cls_local) <= tau_cls)
#     memb = np.column_stack([
#         (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_cls)
#         for j in range(P_child.shape[1])
#     ])
#     cls_set_size_mondrian[mask] = memb.sum(axis=1)

# # Hierarchical propagated membership along TRUE path
# div_in_set_hier = sec_in_set & div_in_set_mondrian
# cls_in_set_hier = div_in_set_hier & cls_in_set_mondrian

# # Output-level stats (exact for this tiny hierarchy)
# cls_size_by_div = {}
# for d, child in classes.items():
#     child_idx = np.array([idx_cls_dict[c] for c in child], dtype=int)
#     P_child = renorm_rows(P_cls_cal[:, child_idx])
#     memb = np.column_stack([
#         (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_cls)
#         for j in range(P_child.shape[1])
#     ])
#     cls_size_by_div[d] = memb.sum(axis=1)

# leaf_union_size = np.zeros(len(X_cal), dtype=int)
# for s, divs_s in divisions.items():
#     s_idx = idx_sec[s]
#     s_in = sec_membership[:, s_idx]
#     add = np.zeros(len(X_cal), dtype=int)
#     for d in divs_s:
#         add += cls_size_by_div[d]
#     leaf_union_size += s_in.astype(int) * add

# reported_level = np.full(len(X_cal), "class_level", dtype=object)
# reported_level[leaf_union_size > MAX_LEAF_SIZE] = "division_level"
# reported_level[rej_cal] = "reject"

# t_exact = time.time() - t0

In [ ]:
# # ------------------------------
# # 7) Print results
# # ------------------------------
# print(f"=== FULLY UNIFIED EXACT METRICS (argsort-identical RAPS), N={N:,} (cal size={len(X_cal):,}) ===")
# print(f"Data generation: {t_gen:.2f}s | Fit: {t_fit:.2f}s | predict_proba: {t_prob:.2f}s | exact-metrics: {t_exact:.2f}s\n")

# print("Thresholds:")
# print("  tau_sec:", tau_sec)
# print("  tau_div:", tau_div)
# print("  tau_cls:", tau_cls)
# print("  tau_rej (1-maxp):", tau_rej, "=> reject if max p_sec <", float(1 - tau_rej))
# print()

# print("Reject stats:")
# print("  reject_rate (all cal):", float(rej_cal.mean()))
# print("  reject_rate on flagged OOD cal points:", float(rej_cal[is_ood_cal].mean()))
# print("  reject_rate on flagged in-dist cal points:", float(rej_cal[~is_ood_cal].mean()))
# print()

# def mean_if(mask, arr_bool):
#     return float(arr_bool[mask].mean()) if np.any(mask) else float("nan")

# def avg_if(mask, arr_num):
#     return float(arr_num[mask].mean()) if np.any(mask) else float("nan")

# print("Coverage (exact set membership):")
# print("  Section coverage (unconditional):", float(sec_in_set.mean()))
# print("  Division coverage (Mondrian, unconditional):", float(div_in_set_mondrian.mean()))
# print("  Class   coverage (Mondrian, unconditional):", float(cls_in_set_mondrian.mean()))
# print("  Division coverage (hierarchical propagated):", float(div_in_set_hier.mean()))
# print("  Class   coverage (hierarchical propagated):", float(cls_in_set_hier.mean()))
# print()

# print("Coverage among KEPT (not rejected):")
# print("  Section coverage | kept:", mean_if(kept, sec_in_set))
# print("  Division Mondrian coverage | kept:", mean_if(kept, div_in_set_mondrian))
# print("  Class   Mondrian coverage | kept:", mean_if(kept, cls_in_set_mondrian))
# print("  Division hierarchical coverage | kept:", mean_if(kept, div_in_set_hier))
# print("  Class   hierarchical coverage | kept:", mean_if(kept, cls_in_set_hier))
# print()

# print("Average set sizes (all cal / kept):")
# print("  |Gamma_sec|:", float(sec_set_size.mean()), "/", avg_if(kept, sec_set_size))
# print("  |Gamma_div(true-parent)|:", float(div_set_size_mondrian.mean()), "/", avg_if(kept, div_set_size_mondrian))
# print("  |Gamma_cls(true-parent)|:", float(cls_set_size_mondrian.mean()), "/", avg_if(kept, cls_set_size_mondrian))
# print()

# print("Backtracking / output-level stats (using MAX_LEAF_SIZE=%d):" % MAX_LEAF_SIZE)
# print("  P(output=reject):", float((reported_level=="reject").mean()))
# print("  P(output=division_level):", float((reported_level=="division_level").mean()))
# print("  P(output=class_level):", float((reported_level=="class_level").mean()))

In [ ]:
sec_labels = sec_clf.config.id2label
div_labels = {k2: v2.config.id2label for k2, v2 in div_clf_dict.items()}
cls_labels = {k2: v2.config.id2label for k2, v2 in cls_clf_dict.items()}

In [ ]:
### Test different tau

#tau_rej = 0.9
# tau_div_dict = {'A': 0.98, 'C': 0.98, 'K': 1}
# tau_cls_dict = {'1': 0.98,
#  '3': 0.98,
#  '21': 0.98,
#  '26': 0.98,
#  '64': 0.98,
#  '66': 0.98}

In [ ]:
K_FREE = 1
LAMBDA = 0.05

def renorm(v):
    s = float(np.sum(v))
    return v/s if s > 0 else np.ones_like(v)/len(v)

def raps_score_1d(p, j, k_free=K_FREE, lam=LAMBDA):
    order = np.argsort(-p)
    r = int(np.where(order == j)[0][0]) + 1
    return float(np.sum(p[order[:r]]) + lam * max(0, r - k_free))

def raps_set_1d(p, labels, tau):
    print("RAPS scores:")
    # for j in range(len(labels)): 
    #     print(p[j], str(labels[j]), raps_score_1d(p, j), tau, raps_score_1d(p, j) <= tau)
    return [str(labels[j]) for j in range(len(labels)) if raps_score_1d(p, j) <= tau]

def reject_score_1d(pS):
    return float(1.0 - np.max(pS))

def predict_full_one(desc, max_leaf_size=3, temperature=1):
    
    # get 1 level predictio

    pS = BERT_classification_chunk([desc], sec_clf, temperature=temperature)[0]
    rs = reject_score_1d(pS)

    sec_set = raps_set_1d(pS, sec_labels, tau_sec)
    sec_probs = {sec_labels[i]: pS[i] for i in range(len(pS))}

    # if empty return nothing
    if rs > tau_rej:
        return {
            "section_set": [],
            "section_probs": sec_probs,
            "division_sets": {},
            "division_probs": {}, 
            "class_probs": {},
            "class_sets": {},
            "final_output": {"type":"reject", "reason":"low section confidence", "reject_score": rs},
        }

    if len(sec_set) == 0:
        sec_set = [str(sec_labels[int(np.argmax(pS))])]

    # Temp 

    sec_set_ = [s for s in sec_set if s in list(div_clf_dict.keys())]
    # get 2 level prediction (of the children)

    div_sets = {}
    div_probs = {}
    #for s in list(set(sec_labels) & set(sec_set)):
    for s in sec_set_: 
        pD = BERT_classification_chunk([desc], div_clf_dict[s], temperature=temperature)[0]
        
        child = list(div_labels[s].values())
        print(child)
        print(idx_div_dict)
        print(idx_div_dict[s])
        p_child = renorm(np.array([pD[idx_div_dict[s][str(d)]] for d in child]))

        dset = raps_set_1d(p_child, child, tau_div_dict[s])
        if len(dset) == 0:
            dset = [child[int(np.argmax(p_child))]]

        div_probs[s] = {div_labels[s][i]: pD[i] for i in range(len(pD))}
        div_sets[s] = dset

    # get 3 level prediction (of the children)

    cls_sets = {}
    cls_probs = {}
    leaf = []
    for ds in div_sets.values():
        for d in ds:
            if d in cls_clf_dict.keys():
                pC = BERT_classification_chunk([desc], cls_clf_dict[d], temperature=2)[0]
                child = list(cls_labels[d].values())
                p_child = renorm(np.array([pC[idx_cls_dict[d][str(c)]] for c in child]))
                cset = raps_set_1d(p_child, child, tau_cls_dict[d])
                if len(cset) == 0:
                    cset = [child[int(np.argmax(p_child))]]
                cls_sets[d] = cset
                cls_probs[d] = {cls_labels[d][i]: pC[i] for i in range(len(pC))}
                leaf.extend(cset)

    leaf = sorted(set(leaf))
    if len(leaf) <= max_leaf_size:
        final = {"type":"class_level", "codes": leaf}
    else:
        div_out = sorted({d for ds in div_sets.values() for d in ds})
        final = {"type":"division_level", "codes": div_out}

    return {
        "section_set": sec_set,        
        "section_probs": sec_probs,
        "division_sets": div_sets,
        "division_probs": div_probs,
        "class_sets": cls_sets,
        "class_probs": cls_probs,
        "final_output": final,
    }

predict_full_one("manufacture metal tanks containers reservoirs pressure vessels")

### Test on Paragraphs

In [ ]:
y_lvl_3_test = list(map(lambda x: "0"+x, y_lvl_3_test))

In [ ]:
false_positives_lvl_1 = []
false_positives_lvl_2 = []
false_positives_lvl_3 = []

true_lvl_1 = []
true_lvl_2 = []
true_lvl_3 = []

for i in tqdm(range(len(X_lvl_3_test))):
    desc = X_lvl_3_test[i]
    # ts, td, tc = y_sec_cal[i], y_div_cal[i], y_cls_cal[i]

    tc = y_lvl_3_test[i]
    td = [div for div, cls_ in classes.items() if tc in cls_][0]
    ts = [sec for sec, divs in divisions.items() if td in divs][0]
    
    pred = predict_full_one(desc, max_leaf_size=3)


    print("\n==================================================")
    print(f"Description {i}:\n ", desc)
    print(f"True: section={ts}, division={td}, class={tc}")
    print("Section set:", pred["section_set"])
    print("Section probs:", pred["section_probs"])
    print("Division sets:", pred["division_sets"])
    print("Division probs:", pred["division_probs"])
    print("Class sets:", pred["class_sets"])
    print("Class probs:", pred["class_probs"])
    print("Final output:", pred["final_output"])   

    # How good is the prediction
    print("Evaluation:")     
    true = (ts in pred["section_set"])
    true_lvl_1.append(true)
    print("Right class in Level 1?", true)
    print("Level 1 Empty?", (len(pred["section_set"])==0))
    fp = not(ts in pred["section_set"]) and not(len(pred["section_set"])==0)
    false_positives_lvl_1.append(fp)
    print("False Positive: ", fp)
    print()

    divisions_list = [x for k, v in pred["division_sets"].items() for x in v]
    true = (td in divisions_list)
    true_lvl_2.append(true)
    #print("Right class in Level 2 or Level 2 empty? ", (td in divisions_list) or (len(pred["division_sets"])==0))
    print("Right class in Level 2?", true)
    print("Level 2 Empty?", (len(pred["division_sets"])==0))
    fp = not(td in divisions_list) and not(len(pred["division_sets"])==0)
    false_positives_lvl_2.append(fp)
    print("False Positive: ", fp)
    print()

    class_list = [x for k, v in pred["class_sets"].items() for x in v]
    true = (tc in class_list)
    true_lvl_3.append(true)
    #print("Right class in Level 3 or Level 3 empty? ", (tc in class_list) or (len(pred["class_sets"])==0))
    print("Right class in Level 3?", true)
    print("Level 3 Empty?", (len(pred["class_sets"])==0))
    fp = not(tc in class_list) and not(len(pred["class_sets"])==0)
    false_positives_lvl_3.append(fp)
    print("False Positive: ", fp)
    print()

In [ ]:
print("false_positives_lvl_1: ", np.mean(false_positives_lvl_1))
print("false_positives_lvl_2: ", np.mean(false_positives_lvl_2))
print("false_positives_lvl_3: ", np.mean(false_positives_lvl_3))

print("true_lvl_1: ", np.mean(true_lvl_1))
print("true_lvl_2: ", np.mean(true_lvl_2))
print("true_lvl_3: ", np.mean(true_lvl_3)) 

print("Nbr of test datapoints: ", len(true_lvl_2))

## Test for Desc Pages

In [ ]:
# use description pages

dataset_path = "data/datasets/stoxx_600"

over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

nace_classes_full = pd.read_csv(over_view_df_path, index_col=0, sep=";")



In [ ]:
nace_classes = nace_classes_full[nace_classes_full["description_page"].notna()]
nace_classes

In [ ]:
description_page_path = "data/datasets/stoxx_600/company_descriptions_txt/"

In [ ]:
nace_classes["NACE_lvl_3"] = nace_classes["NACE"].apply(lambda x: NACE_helper.get_all_level(x)[3])
nace_classes = nace_classes[nace_classes["NACE_lvl_3"].apply(lambda x: x in all_classes)]

In [ ]:
nace_classes = nace_classes.sort_values(by="NACE_letter")
nace_classes

In [ ]:
false_positives_lvl_1 = []
false_positives_lvl_2 = []
false_positives_lvl_3 = []

true_lvl_1 = []
true_lvl_2 = []
true_lvl_3 = []

for i, row in nace_classes.iterrows():
#for i, row in nace_classes.loc[460:460].iterrows():
#for i in range(len(X_lvl_3_test)):
    with open(os.path.join(description_page_path,row["Report"].replace("pdf", "txt")), "r") as f: 
        desc = f.read()

    # ts, td, tc = y_sec_cal[i], y_div_cal[i], y_cls_cal[i]

    tc = row["NACE_lvl_3"]
    td = [div for div, cls_ in classes.items() if tc in cls_][0]
    ts = [sec for sec, divs in divisions.items() if td in divs][0]
    
    pred = predict_full_one(desc, max_leaf_size=3)

    y_lvl_3_test

    print("\n==================================================")
    print(f"Description {i}:\n ", desc.replace("\n", " "))
    print(f"True: section={ts}, division={td}, class={tc}")
    print("Section probs:", pred["section_probs"])
    print("Section set:", pred["section_set"])
    print()

    print("Division probs:", pred["division_probs"])
    print("Division sets:", pred["division_sets"])
    print()

    print("Class probs:", pred["class_probs"])
    print("Class sets:", pred["class_sets"])
    print()
    print("Final output:", pred["final_output"])   
    print()

    # How good is the prediction
    print("Evaluation:")     
    true = (ts in pred["section_set"])
    true_lvl_1.append(true)
    print("Right class in Level 1?", true)
    print("Level 1 Empty?", (len(pred["section_set"])==0))
    fp = not(ts in pred["section_set"]) and not(len(pred["section_set"])==0)
    false_positives_lvl_1.append(fp)
    print("False Positive: ", fp)
    print()

    divisions_list = [x for k, v in pred["division_sets"].items() for x in v]
    true = (td in divisions_list)
    true_lvl_2.append(true)
    #print("Right class in Level 2 or Level 2 empty? ", (td in divisions_list) or (len(pred["division_sets"])==0))
    print("Right class in Level 2?", true)
    print("Level 2 Empty?", (len(pred["division_sets"])==0))
    fp = not(td in divisions_list) and not(len(pred["division_sets"])==0)
    false_positives_lvl_2.append(fp)
    print("False Positive: ", fp)
    print()

    class_list = [x for k, v in pred["class_sets"].items() for x in v]
    true = (tc in class_list)
    true_lvl_3.append(true)
    #print("Right class in Level 3 or Level 3 empty? ", (tc in class_list) or (len(pred["class_sets"])==0))
    print("Right class in Level 3?", true)
    print("Level 3 Empty?", (len(pred["class_sets"])==0))
    fp = not(tc in class_list) and not(len(pred["class_sets"])==0)
    false_positives_lvl_3.append(fp)
    print("False Positive: ", fp)
    print()

In [ ]:
print("false_positives_lvl_1: ", np.mean(false_positives_lvl_1))
print("false_positives_lvl_2: ", np.mean(false_positives_lvl_2))
print("false_positives_lvl_3: ", np.mean(false_positives_lvl_3))
print("true_lvl_1: ", np.mean(true_lvl_1))
print("true_lvl_2: ", np.mean(true_lvl_2))
print("true_lvl_3: ", np.mean(true_lvl_3))


### With llm summaries

Please summarize the business model in 4 sentences: 

In [ ]:
description_page_path_llm_summary = "data/datasets/stoxx_600/company_descriptions_txt_1/"
# from_ = os.path.join(description_page_path,row["Report"].replace("pdf", "txt"))
# to_ = os.path.join(description_page_path_to,row["Report"].replace("pdf", "txt"))

# import shutil

# for i, row in nace_classes.iterrows():
#     from_ = os.path.join(description_page_path,row["Report"].replace("pdf", "txt"))
#     to_ = os.path.join(description_page_path_llm_summary,row["Report"].replace("pdf", "txt"))
#     shutil.copyfile(from_, to_)

In [ ]:
nace_classes

In [ ]:
false_positives_lvl_1 = []
false_positives_lvl_2 = []
false_positives_lvl_3 = []
true_lvl_1 = []
true_lvl_2 = []
true_lvl_3 = []

for i, row in nace_classes.iterrows():
#for i, row in nace_classes.loc[460:460].iterrows():
#for i in range(len(X_lvl_3_test)):
    try: 
        with open(os.path.join(description_page_path_llm_summary,row["Report"].replace("pdf", "txt")), "r") as f: 
            desc = f.read()
    except FileNotFoundError:
        print("File not found:", os.path.join(description_page_path_llm_summary,row["Report"].replace("pdf", "txt")))
        continue

    # ts, td, tc = y_sec_cal[i], y_div_cal[i], y_cls_cal[i]

    tc = row["NACE_lvl_3"]
    td = [div for div, cls_ in classes.items() if tc in cls_][0]
    ts = [sec for sec, divs in divisions.items() if td in divs][0]
    
    print("\n==================================================")
    print(f"Description {i}:\n ", desc.replace("\n", " "))
    
    pred = predict_full_one(desc, max_leaf_size=3, temperature=1)


    print(f"True: section={ts}, division={td}, class={tc}")
    print("Section probs:", pred["section_probs"])
    print("Section set:", pred["section_set"])
    print()

    print("Division probs:", pred["division_probs"])
    print("Division sets:", pred["division_sets"])
    print()

    print("Class probs:", pred["class_probs"])
    print("Class sets:", pred["class_sets"])
    print()
    print("Final output:", pred["final_output"])   
    print()

    # How good is the prediction
    print("Evaluation:")     
    true = (ts in pred["section_set"])
    true_lvl_1.append(true)
    print("Right class in Level 1?", true)
    print("Level 1 Empty?", (len(pred["section_set"])==0))
    fp = not(ts in pred["section_set"]) and not(len(pred["section_set"])==0)
    false_positives_lvl_1.append(fp)
    print("False Positive: ", fp)
    print()

    divisions_list = [x for k, v in pred["division_sets"].items() for x in v]
    true = (td in divisions_list)
    true_lvl_2.append(true)
    #print("Right class in Level 2 or Level 2 empty? ", (td in divisions_list) or (len(pred["division_sets"])==0))
    print("Right class in Level 2?", true)
    print("Level 2 Empty?", (len(pred["division_sets"])==0))
    fp = not(td in divisions_list) and not(len(pred["division_sets"])==0)
    false_positives_lvl_2.append(fp)
    print("False Positive: ", fp)
    print()

    class_list = [x for k, v in pred["class_sets"].items() for x in v]
    true = (tc in class_list)
    true_lvl_3.append(true)
    #print("Right class in Level 3 or Level 3 empty? ", (tc in class_list) or (len(pred["class_sets"])==0))
    print("Right class in Level 3?", true)
    print("Level 3 Empty?", (len(pred["class_sets"])==0))
    fp = not(tc in class_list) and not(len(pred["class_sets"])==0)
    false_positives_lvl_3.append(fp)
    print("False Positive: ", fp)
    print()

In [ ]:
len(false_positives_lvl_1)

In [ ]:
# Temperatur bei calibrierugn auf 2 angepasst!

print("False Positives in Level 1: ", np.mean(false_positives_lvl_1))
print("False Positives in Level 2: ", np.mean(false_positives_lvl_2))
print("False Positives in Level 3: ", np.mean(false_positives_lvl_3))
print("True Predictions in Level 1: ", np.mean(true_lvl_1))
print("True Predictions in Level 2: ", np.mean(true_lvl_2))
print("True Predictions in Level 3: ", np.mean(true_lvl_3))

In [ ]:
nace_classes_full
nace_classes_full["NACE_lvl_3"] = nace_classes_full["NACE"].apply(lambda x: NACE_helper.get_all_level(x)[3])
nace_classes_full = nace_classes_full[nace_classes_full["NACE_lvl_3"].apply(lambda x: x in all_classes)]
nace_classes_full

In [ ]:
df_overview = pd.read_csv("data/datasets/reports_subset_from_full_data_3/reports_subset_from_full_data_3_overview.csv", index_col=0)
df_overview["NACE_lvl_3"] = df_overview["NACE"].apply(lambda x: NACE_helper.get_all_level(x)[3])
df_overview.groupby("NACE_lvl_3").count()